# BENCHMARK TOÀN DIỆN RESNET50 — NEWDATASET

Chỉ huấn luyện/đánh giá các thư mục lớp trong `NewDataset`; không gộp Oxford-IIIT Pet.
Notebook độc lập, giữ thứ tự và các phương pháp từ `benchmark_resnet50.ipynb`.

**Kaggle:** Import notebook này; Add Input dataset chứa `NewDataset/<giống>/<ảnh>`; bật GPU và Internet.
**MacBook:** cài dependencies ở cell dưới, chọn kernel Python của môi trường; tự chọn CUDA → MPS → CPU.
Đường dẫn `DATA_ROOT` phải trỏ tới thư mục chứa trực tiếp các thư mục giống.

**Thiết kế:** loại ảnh trùng pixel trước khi chia, giữ 20% mỗi lớp làm test; 80% còn lại là trainval.
Trainval chia 9:1 theo lớp. Full trainval dùng toàn bộ 80%, không dùng test để chọn tham số.
Ảnh gần trùng hoặc cùng một cá thể không được phát hiện bởi hash; cần kiểm tra nguồn dữ liệu nếu muốn split theo cá thể.
Cache tách theo fingerprint dữ liệu và backbone; mọi output ghi vào thư mục có quyền ghi.
Các số liệu và biểu đồ sẽ được tính khi chạy, không sao chép kết quả Oxford cũ.


In [ ]:
%pip install "transformers>=4.45,<5" "opencv-python-headless>=4.8,<5" scikit-learn matplotlib seaborn pandas tabulate tqdm
# macOS only, if torch/torchvision are missing: %pip install torch torchvision


In [ ]:
import os
import time
import json
import random
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
import torchvision.models as models
from torchvision.datasets import OxfordIIITPet

# Thiết lập thẩm mỹ đồ thị chuẩn khoa học
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 1.0

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    elif torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

# Cấu hình thiết bị phần cứng
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"🚀 Thiết bị tính toán được cấu hình: {device}")
print(f"📦 Phiên bản PyTorch: {torch.__version__} | Torchvision: {torchvision.__version__}")
RUN_NAME = 'resnet50'

from pathlib import Path
def as_feature_tensor(value):
    # Transformers 4 returns a tensor; newer APIs may return a pooled model output.
    return value if isinstance(value, torch.Tensor) else value.pooler_output

DATA_ROOT = None  # e.g. "/kaggle/input/animal-newdataset/NewDataset"
SEED = 42
TEST_RATIO = 0.20
BATCH_SIZE = 32  # reduce to 16/8 if GPU/MPS memory is limited
TIP_EPOCHS = 20
ADAPTER_EPOCHS = 50
COOP_EPOCHS = 50
RESNET_EPOCHS = 30
OUTPUT_ROOT = Path('/kaggle/working/newdataset_benchmark') if Path('/kaggle').exists() else Path('results/newdataset_benchmark')
OUTPUT_ROOT = OUTPUT_ROOT.resolve()
output_dir = OUTPUT_ROOT / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
"""Runtime reports: measured results only, with explicit latency scope."""
def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()


def measure_latency_ms(fn, num_runs=50, warmup=5):
    with torch.no_grad():
        for _ in range(warmup):
            fn()
        synchronize()
        times = []
        for _ in range(num_runs):
            synchronize()
            start = time.perf_counter()
            fn()
            synchronize()
            times.append((time.perf_counter() - start) * 1000)
    return {key: float(value) for key, value in {
        'mean': np.mean(times), 'median': np.median(times),
        'p95': np.percentile(times, 95), 'p99': np.percentile(times, 99)}.items()}


def save_checkpoint(name, model, **extra):
    # CoOp saves only the learned context; pretrained CLIP is shared and frozen.
    state = {'ctx': model.ctx.detach().cpu()} if hasattr(model, 'ctx') else {
        key: value.detach().cpu() for key, value in model.state_dict().items()}
    torch.save({'state_dict': state, 'classes': class_names,
                'fingerprint': manifest['fingerprint'], 'seed': SEED,
                'backbone': MODEL_ID,
                'alpha': getattr(model, 'alpha', None), 'beta': getattr(model, 'beta', None),
                'logit_scale': globals().get('logit_scale_val'), **extra}, output_dir / f'{name}.pt')


def build_report(results, predictors, histories, encoder):
    import pandas as pd
    from IPython.display import display
    frame = pd.DataFrame(results.values())
    base = float(frame.iloc[0]['Test_Acc'])
    frame['Delta_vs_Baseline'] = frame['Test_Acc'] - base
    frame['Head_90split_Latency_ms'] = None
    latency = {}
    for name, predictor in predictors.items():
        stats = measure_latency_ms(lambda: predictor(test_features_raw[:1].to(device)))
        latency[name] = stats
        frame.loc[frame.Method == name, 'Head_90split_Latency_ms'] = stats['mean']
    # Encoder timing uses synthetic pixels already on device; excludes PIL, OpenCV and IO.
    pixels = torch.zeros(1, 3, 224, 224, device=device)
    latency['encoder_raw'] = measure_latency_ms(lambda: encoder(pixels))
    latency['encoder_5_views'] = measure_latency_ms(lambda: encoder(pixels.expand(5, -1, -1, -1)))
    frame.to_csv(output_dir / 'benchmark_results.csv', index=False)
    display(frame)
    summary = {'dataset': manifest['fingerprint'], 'classes': class_names,
               'device': str(device), 'backbone': MODEL_ID, 'seed': SEED,
               'latency_scope': 'Representative 9:1 head on cached raw features, repeated per method in the table; encoder on device-resident synthetic pixels. Not end-to-end.',
               'latency_ms': latency, 'results': list(results.values())}
    (output_dir / 'benchmark_results.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    (output_dir / 'training_history.json').write_text(json.dumps(histories, indent=2), encoding='utf-8')
    (output_dir / 'environment.json').write_text(json.dumps({
        'torch': torch.__version__, 'torchvision': torchvision.__version__,
        'transformers': __import__('transformers').__version__, 'device': str(device),
        'gpu': torch.cuda.get_device_name() if device.type == 'cuda' else None}, indent=2), encoding='utf-8')

    def finish(name):
        plt.tight_layout()
        plt.savefig(output_dir / f'{name}.png', dpi=160, bbox_inches='tight')
        plt.show()

    plt.figure(figsize=(13, 6))
    sns.barplot(data=frame, x='Method', y='Test_Acc', hue='Scenario')
    plt.xticks(rotation=18, ha='right'); plt.ylabel('Test Accuracy (%)')
    finish('01_accuracy')
    for metric, title, filename in [('loss', 'Training Loss', '02_loss'),
                                     ('val_acc', 'Validation Accuracy (%)', '03_validation')]:
        plt.figure(figsize=(10, 5))
        for name, history in histories.items():
            values = history[metric]
            if any(v is not None for v in values):
                plt.plot(range(1, len(values) + 1), values, label=name)
        plt.xlabel('Epoch'); plt.ylabel(title); plt.legend()
        finish(filename)
    plt.figure(figsize=(11, 5))
    plt.bar(latency.keys(), [v['mean'] for v in latency.values()])
    plt.xticks(rotation=25, ha='right'); plt.ylabel('Measured latency (ms; scope in JSON)')
    finish('04_latency')
    plt.figure(figsize=(11, 5))
    sns.barplot(data=frame, x='Method', y='Trainable_Params', hue='Scenario')
    plt.xticks(rotation=20, ha='right'); plt.ylabel('Trainable parameters')
    finish('05_parameters')
    # Select only by validation accuracy; do not choose the winning method on test.
    candidates = frame[frame['Val_Acc'].notna() & frame.Method.isin(predictors)]
    selected = candidates.sort_values('Val_Acc', ascending=False, kind='stable').iloc[0]['Method']
    with torch.no_grad():
        predictor = predictors[selected]
        preds = predictor(test_features_tta.to(device)).argmax(-1).cpu().numpy()
        raw_preds = predictor(test_features_raw.to(device)).argmax(-1).cpu().numpy()
    truth = test_labels.cpu().numpy()
    matrix = confusion_matrix(truth, preds, labels=list(range(num_classes)))
    plt.figure(figsize=(14, 12))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{selected}: validation-selected, TTA, 9:1 support'); plt.xlabel('Predicted'); plt.ylabel('True')
    finish('06_confusion_matrix')
    gains = [(100 * (preds[truth == i] == i).mean() - 100 * (raw_preds[truth == i] == i).mean())
             for i in range(num_classes)]
    order = np.argsort(gains)[-10:]
    plt.figure(figsize=(10, 6))
    plt.barh([class_names[i] for i in order], np.array(gains)[order])
    plt.xlabel('TTA gain vs raw (percentage points), same validation-selected head')
    finish('07_per_class_tta_gain')
    report = classification_report(truth, preds, target_names=class_names, output_dict=True, zero_division=0)
    (output_dir / 'classification_report.json').write_text(json.dumps({
        'selected_by_validation': selected, 'report': report,
        'confusion_matrix': matrix.tolist()}, indent=2), encoding='utf-8')
    rows = frame.fillna('—').astype(str)
    table = '| ' + ' | '.join(rows.columns) + ' |\n'
    table += '| ' + ' | '.join(['---'] * len(rows.columns)) + ' |\n'
    table += '\n'.join('| ' + ' | '.join(row) + ' |' for row in rows.to_numpy())
    (output_dir / 'benchmark_report.md').write_text(
        f'# {MODEL_ID} — NewDataset\n\n{table}\n\n'
        'Latency: head-only and encoder-only, excluding image IO and preprocessing. '
        'Full trainval has no independent validation score.\n', encoding='utf-8')
    # Compare only with a matching split fingerprint, never legacy Oxford scores.
    sibling = output_dir.parent / ('resnet50' if output_dir.name == 'clip' else 'clip') / 'benchmark_results.json'
    if sibling.exists():
        other = json.loads(sibling.read_text(encoding='utf-8'))
        if other['dataset'] == manifest['fingerprint']:
            combined = pd.concat([frame.assign(Backbone=MODEL_ID),
                                  pd.DataFrame(other['results']).assign(Backbone=other['backbone'])])
            combined.to_csv(output_dir / 'clip_vs_resnet50.csv', index=False)
            display(combined[['Backbone', 'Method', 'Scenario', 'Val_Acc', 'Test_Acc']])
        else:
            print('Skipped comparison: different split fingerprints')
    print(f'Artifacts saved to {output_dir}')


In [ ]:
"""Image-folder dataset, content deduplication and reproducible benchmark splits."""
import hashlib
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

from PIL import Image, ImageOps
from torch.utils.data import Dataset

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}


def discover_dataset(explicit=None):
    if explicit:
        root = Path(explicit).expanduser()
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root.resolve()
    candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for name in ('NewDataset', 'NewDatasett'):
            candidate = parent / 'dataset' / name
            if candidate.is_dir():
                candidates.append(candidate.resolve())
    kaggle = Path('/kaggle/input')
    if kaggle.exists():
        candidates.extend(p.resolve() for p in kaggle.rglob('*')
                          if p.is_dir() and p.name in ('NewDataset', 'NewDatasett'))
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise ValueError(f'Set DATA_ROOT to the folder containing breed folders; found {candidates}')
    return candidates[0]


class BreedDataset(Dataset):
    def __init__(self, root, records, classes):
        self.root, self.records, self.classes = Path(root), records, classes

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        item = self.records[index]
        with Image.open(self.root / item['path']) as image:
            image = ImageOps.exif_transpose(image).convert('RGB')
        return image, item['label']


def prepare_dataset(root, output_dir, seed=42, test_ratio=0.2):
    """Split unique decoded images per class; reject conflicting duplicate labels."""
    root, output_dir = Path(root), Path(output_dir)
    if not 0 < test_ratio < 1:
        raise ValueError('test_ratio must be between 0 and 1')
    classes = sorted(p.name for p in root.iterdir() if p.is_dir() and not p.name.startswith('.'))
    if len(classes) < 2:
        raise ValueError('Expected at least two breed folders')
    groups, seen, duplicates, invalid = defaultdict(list), {}, [], []
    for label, name in enumerate(classes):
        for path in sorted((root / name).rglob('*')):
            if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            relative = path.relative_to(root).as_posix()
            try:
                with Image.open(path) as image:
                    image = ImageOps.exif_transpose(image).convert('RGB')
                    digest = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
            except (OSError, ValueError) as error:
                invalid.append({'path': relative, 'error': str(error)})
                continue
            if digest in seen:
                previous = seen[digest]
                if previous['label'] != label:
                    raise ValueError(f'Identical image has conflicting labels: {previous["path"]}, {relative}')
                duplicates.append({'path': relative, 'same_as': previous['path']})
                continue
            record = {'path': relative, 'label': label, 'sha256': digest}
            seen[digest] = record
            groups[label].append(record)
    trainval, test = [], []
    for label, name in enumerate(classes):
        items = groups[label][:]
        if len(items) < 4:
            raise ValueError(f'{name}: need >= 4 valid unique images for train/val/test')
        random.Random(seed + label).shuffle(items)
        count = min(len(items) - 2, max(1, round(len(items) * test_ratio)))
        test.extend(items[:count])
        trainval.extend(items[count:])
    manifest = {'seed': seed, 'test_ratio': test_ratio, 'classes': classes,
                'trainval': trainval, 'test': test, 'duplicates_removed': duplicates,
                'invalid_images': invalid, 'version': 1}
    fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    manifest['fingerprint'] = fingerprint
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'split_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    (output_dir / 'classes.json').write_text(json.dumps(classes, indent=2), encoding='utf-8')
    print(f'{len(classes)} classes | trainval={len(trainval)} | test={len(test)}')
    print(f'Removed duplicates={len(duplicates)}; invalid images={len(invalid)}; fingerprint={fingerprint}')
    print({classes[k]: v for k, v in sorted(Counter(r['label'] for r in trainval + test).items())})
    return BreedDataset(root, trainval, classes), BreedDataset(root, test, classes), manifest

dataset_root = discover_dataset(DATA_ROOT)
pets_trainval, pets_test, manifest = prepare_dataset(dataset_root, output_dir, SEED, TEST_RATIO)
class_names = pets_trainval.classes
num_classes = len(class_names)
trainval_labels = torch.tensor([r['label'] for r in pets_trainval.records], dtype=torch.long)
test_labels = torch.tensor([r['label'] for r in pets_test.records], dtype=torch.long)

MODEL_ID = 'torchvision/resnet50/IMAGENET1K_V2'


## 1. QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU TỐI ƯU & TTA 5 GÓC NHÌN (5-VIEW TTA)

---

Kế thừa từ quy trình tiền xử lý phân tích nâng cao:
1. **Lọc song phương (Bilateral Filter)**: Bảo toàn biên kết cấu lông thú (*fur texture*), mắt và râu ($d=7, \sigma_{\text{color}}=50, \sigma_{\text{space}}=50$).
2. **CLAHE trên không gian màu LAB**: Cân bằng tương phản cục bộ kênh sáng $L$ ($clipLimit=2.0$) mà không làm sai lệch sắc độ $A, B$.
3. **Quy trình TTA 5 góc nhìn**:
   - **View 1**: Ảnh gốc (Original)
   - **View 2**: Lọc song phương (Bilateral Denoise)
   - **View 3**: Cân bằng CLAHE LAB
   - **View 4**: Tiền xử lý kết hợp tối ưu (Bilateral + CLAHE)
   - **View 5**: Tăng cường hình học (Optimal + Flip Left-Right + Rotate $15^\circ$)

Đặc trưng TTA thu được bằng cách lấy trung bình cộng đặc trưng của 5 góc nhìn:
$$\mathbf{f}_{\text{TTA}} = \text{Normalize}\left(\frac{1}{5}\sum_{i=1}^5 \mathbf{f}_{\text{view}_i}\right)$$

In [ ]:
# Khởi tạo kiến trúc ResNet-50 Pretrained
weights = models.ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

resnet_full = models.resnet50(weights=weights)
backbone = nn.Sequential(*list(resnet_full.children())[:-1]).to(device)
backbone.eval()
backbone.requires_grad_(False)

# Định nghĩa các hàm biến đổi hình ảnh
def apply_denoise(pil_img):
    img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    denoised_bgr = cv2.bilateralFilter(img_bgr, d=7, sigmaColor=50, sigmaSpace=50)
    return Image.fromarray(cv2.cvtColor(denoised_bgr, cv2.COLOR_BGR2RGB))

def apply_clahe(pil_img):
    img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    clahe_bgr = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
    return Image.fromarray(cv2.cvtColor(clahe_bgr, cv2.COLOR_BGR2RGB))

def optimal_preprocessing_pipeline(pil_img):
    img_bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    denoised_bgr = cv2.bilateralFilter(img_bgr, d=7, sigmaColor=50, sigmaSpace=50)
    lab = cv2.cvtColor(denoised_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    clahe_bgr = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
    return Image.fromarray(cv2.cvtColor(clahe_bgr, cv2.COLOR_BGR2RGB))

transform_geom = lambda img: optimal_preprocessing_pipeline(img).transpose(Image.Transpose.FLIP_LEFT_RIGHT).rotate(15)

# Lớp Dataset biến đổi ảnh tuần tự an toàn
class ResNetTransformDataset(Dataset):
    def __init__(self, base_ds, transform_fn=None):
        self.base_ds = base_ds
        self.transform_fn = transform_fn
    def __len__(self):
        return len(self.base_ds)
    def __getitem__(self, idx):
        img, lbl = self.base_ds[idx]
        if self.transform_fn is not None:
            img = self.transform_fn(img)
        return preprocess(img), lbl

def extract_resnet_features(base_dataset, transform_fn=None, batch_size=128, desc="Trích xuất đặc trưng"):
    ds = ResNetTransformDataset(base_dataset, transform_fn)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    feats, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc):
            x = x.to(device)
            f = backbone(x).squeeze(-1).squeeze(-1)
            feats.append(f.cpu())
            labels.append(y)
    return torch.cat(feats, dim=0), torch.cat(labels, dim=0)


cache_dir = output_dir / 'cache' / manifest['fingerprint'] / 'resnet50_imagenet_v2_preprocess_v1'
cache_dir.mkdir(parents=True, exist_ok=True)
def get_features(name, dataset, transform=None):
    path = cache_dir / f'{name}.pt'
    if path.exists():
        features = torch.load(path, map_location='cpu', weights_only=True)
        assert features.shape == (len(dataset), 2048)
        return features
    features, _ = extract_resnet_features(dataset, transform_fn=transform, batch_size=BATCH_SIZE, desc=name)
    torch.save(features, path)
    return features
def tta_features(prefix, dataset, raw):
    return torch.stack([raw, get_features(prefix + '_denoise', dataset, apply_denoise),
        get_features(prefix + '_clahe', dataset, apply_clahe),
        get_features(prefix + '_optimal', dataset, optimal_preprocessing_pipeline),
        get_features(prefix + '_geom', dataset, transform_geom)]).mean(0)
trainval_features_raw = get_features('trainval_raw', pets_trainval)
test_features_raw = get_features('test_raw', pets_test)
trainval_features_tta = tta_features('trainval', pets_trainval, trainval_features_raw)
test_features_tta = tta_features('test', pets_test, test_features_raw)
print(trainval_features_raw.shape, test_features_tta.shape)


In [ ]:
# Trực quan hóa ảnh mẫu và các góc nhìn trong quy trình TTA 5 góc nhìn
sample_idx = min(10, len(pets_test) - 1)
sample_img, sample_lbl = pets_test[sample_idx]

view_orig = sample_img
view_denoise = apply_denoise(sample_img)
view_clahe = apply_clahe(sample_img)
view_optimal = optimal_preprocessing_pipeline(sample_img)
view_geom = transform_geom(sample_img)

fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
views_info = [
    (view_orig, f"1. Gốc (Original) - {class_names[sample_lbl]}"),
    (view_denoise, "2. Bilateral Denoise"),
    (view_clahe, "3. CLAHE (LAB)"),
    (view_optimal, "4. Tiền Xử Lý Tối Ưu"),
    (view_geom, "5. Tăng Cường Hình Học")
]

for ax, (v_img, title) in zip(axes, views_info):
    ax.imshow(v_img)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. CHIẾN LƯỢC PHÂN CHIA DỮ LIỆU: TRAIN:VAL (9:1) & FULL TRAINVAL

20% ảnh duy nhất mỗi lớp được giữ làm test từ đầu. Chia 80% trainval còn lại theo lớp:
90% để train/support, 10% validation chọn tham số và checkpoint có validation accuracy cao nhất.
Full trainval không có validation độc lập; chạy đúng số epochs đã cấu hình.
Đây là chọn checkpoint tốt nhất, không phải dừng sớm. CLIP train trên raw features;
ResNet có cả raw và TTA train features như benchmark gốc.


In [ ]:

# Identical raw/TTA and CLIP/ResNet split indices; at least one validation sample/class.
train_indices, val_indices = [], []
generator = torch.Generator().manual_seed(SEED)
for c in range(num_classes):
    indices = (trainval_labels == c).nonzero(as_tuple=True)[0]
    indices = indices[torch.randperm(len(indices), generator=generator)]
    n_train = min(len(indices) - 1, max(1, int(len(indices) * 0.9)))
    train_indices.append(indices[:n_train]); val_indices.append(indices[n_train:])
train_idx, val_idx = torch.cat(train_indices), torch.cat(val_indices)
(output_dir / 'train_val_indices.json').write_text(json.dumps({
    'train': train_idx.tolist(), 'val': val_idx.tolist(), 'fingerprint': manifest['fingerprint']}, indent=2), encoding='utf-8')
print(f'Train={len(train_idx)} | Validation={len(val_idx)} | Test={len(test_labels)}')

X_tr_tta_90, y_tr_90 = trainval_features_tta[train_idx], trainval_labels[train_idx]
X_val_tta_90, y_val_90 = trainval_features_tta[val_idx], trainval_labels[val_idx]
X_tr_raw_90, X_val_raw_90 = trainval_features_raw[train_idx], trainval_features_raw[val_idx]
benchmark_resnet_results = {}


## 3. PHƯƠNG PHÁP 1: RESNET-50 PROTOTYPE CLASSIFIER (NEAREST CENTROID / TRAINING-FREE)

---

Để thiết lập một mốc quy chiếu không qua tối ưu hóa gradient (Training-Free Benchmark) tương đương Tip-Adapter của CLIP:
- Tính toán vector trọng tâm (Prototype) cho mỗi lớp $c$:
  $$\mathbf{c}_k = \frac{1}{|S_k|} \sum_{i \in S_k} \frac{\mathbf{f}_i}{\|\mathbf{f}_i\|_2}, \quad \mathbf{p}_k = \frac{\mathbf{c}_k}{\|\mathbf{c}_k\|_2}$$
- Phân loại mẫu kiểm tra thông qua độ tương đồng Cosine cực đại:
  $$\hat{y} = \arg\max_{k \in \{1, \dots, 37\}} \left(\frac{\mathbf{f}_{\text{test}}}{\|\mathbf{f}_{\text{test}}\|_2} \cdot \mathbf{p}_k\right)$$

In [ ]:
def evaluate_prototype(train_feats, train_lbls, test_feats, test_lbls, val_feats=None, val_lbls=None):
    train_norm = train_feats / train_feats.norm(dim=-1, keepdim=True)
    test_norm = test_feats / test_feats.norm(dim=-1, keepdim=True)
    
    # Tính prototypes
    prototypes = []
    for c in range(num_classes):
        mask = (train_lbls == c)
        proto = train_norm[mask].mean(dim=0)
        proto = proto / proto.norm()
        prototypes.append(proto)
    prototypes = torch.stack(prototypes).to(device)
    
    val_acc = None
    if val_feats is not None and val_lbls is not None:
        val_norm = (val_feats / val_feats.norm(dim=-1, keepdim=True)).to(device)
        sim_val = val_norm @ prototypes.t()
        preds_val = sim_val.argmax(dim=-1)
        val_acc = (preds_val == val_lbls.to(device)).float().mean().item() * 100
        
    sim_test = test_norm.to(device) @ prototypes.t()
    preds_test = sim_test.argmax(dim=-1)
    test_acc = (preds_test == test_lbls.to(device)).float().mean().item() * 100
    
    return val_acc, test_acc

# 1.1. Raw Image (Full Trainval)
_, acc_proto_raw = evaluate_prototype(trainval_features_raw, trainval_labels, test_features_raw, test_labels)

# 1.2. TTA 5-view (9:1 Split)
val_acc_proto_90, acc_proto_tta_90 = evaluate_prototype(X_tr_tta_90, y_tr_90, test_features_tta, test_labels, X_val_tta_90, y_val_90)

# 1.3. TTA 5-view (Full Trainval)
_, acc_proto_tta_full = evaluate_prototype(trainval_features_tta, trainval_labels, test_features_tta, test_labels)

benchmark_resnet_results["1. ResNet-50 (Prototype) | Raw Image (Full Trainval)"] = {
    "Method": "ResNet-50 (Prototype)", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_proto_raw, "Trainable_Params": 0
}
benchmark_resnet_results["1. ResNet-50 (Prototype) | TTA 5-view (9:1 Split)"] = {
    "Method": "ResNet-50 (Prototype)", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_proto_90, "Test_Acc": acc_proto_tta_90, "Trainable_Params": 0
}
benchmark_resnet_results["1. ResNet-50 (Prototype) | TTA 5-view (Full Trainval)"] = {
    "Method": "ResNet-50 (Prototype)", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_proto_tta_full, "Trainable_Params": 0
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ PHƯƠNG PHÁP 1: RESNET-50 PROTOTYPE (TRAINING-FREE)")
print("="*75)
print(f"  1.1. Raw Image (Full Trainval):      {acc_proto_raw:.2f}%")
print(f"  1.2. TTA 5-view (9:1 Split):         {acc_proto_tta_90:.2f}% (Val: {val_acc_proto_90:.2f}%)")
print(f"  1.3. TTA 5-view (Full Trainval):     {acc_proto_tta_full:.2f}%")
print("="*75)

## 4. PHƯƠNG PHÁP 2: RESNET-50 LINEAR PROBE (FROZEN BACKBONE + LINEAR CLASSIFIER HEAD)

---

Quy trình Transfer Learning kinh điển cho Convolutional Networks:
- Toàn bộ 49 tầng tích chập của ResNet-50 được đóng băng hoàn toàn.
- Khởi tạo một tầng tuyến tính mới:
  $$\mathbf{z} = \mathbf{W}_{\text{fc}} \mathbf{f} + \mathbf{b}, \quad \mathbf{W}_{\text{fc}} \in \mathbb{R}^{37 \times 2048}$$
- Huấn luyện với hàm mất mát Cross-Entropy cùng bộ tối ưu **AdamW** ($lr = 10^{-3}, \text{weight\_decay} = 10^{-4}$) và **CosineAnnealingLR** trong 30 epochs.

In [ ]:
def train_linear_probe(train_feats, train_lbls, test_feats, test_lbls, val_feats=None, val_lbls=None, epochs=RESNET_EPOCHS, lr=1e-3):
    classifier = nn.Linear(2048, num_classes).to(device)
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    train_loader = DataLoader(TensorDataset(train_feats.to(device), train_lbls.to(device)), batch_size=128, shuffle=True)
    best_val_acc = 0.0
    best_weights = None
    
    history = {"loss": [], "val_acc": []}
    start_time = time.time()
    for epoch in range(epochs):
        classifier.train()
        total_loss = 0.0
        for x_b, y_b in train_loader:
            optimizer.zero_grad()
            logits = classifier(x_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        history["loss"].append(total_loss / len(train_loader))
        
        if val_feats is not None and val_lbls is not None:
            classifier.eval()
            with torch.no_grad():
                val_preds = classifier(val_feats.to(device)).argmax(dim=-1)
                val_acc = (val_preds == val_lbls.to(device)).float().mean().item() * 100
                history["val_acc"].append(val_acc)
                if val_acc >= best_val_acc:
                    best_val_acc = val_acc
                    best_weights = {k: v.clone() for k, v in classifier.state_dict().items()}
                    
    train_time = time.time() - start_time
    if best_weights is not None:
        classifier.load_state_dict(best_weights)
        
    classifier.eval()
    with torch.no_grad():
        test_preds = classifier(test_feats.to(device)).argmax(dim=-1)
        test_acc = (test_preds == test_lbls.to(device)).float().mean().item() * 100
        
    return best_val_acc if val_feats is not None else None, test_acc, train_time, history, classifier

# 2.1. Raw Image (Full Trainval)
_, acc_lp_raw, t_lp_raw, hist_lp_raw, model_lp_raw = train_linear_probe(trainval_features_raw, trainval_labels, test_features_raw, test_labels)

# 2.2. TTA 5-view (9:1 Split)
val_acc_lp_90, acc_lp_tta_90, t_lp_90, hist_lp_90, model_lp_90 = train_linear_probe(X_tr_tta_90, y_tr_90, test_features_tta, test_labels, X_val_tta_90, y_val_90)

# 2.3. TTA 5-view (Full Trainval)
_, acc_lp_tta_full, t_lp_full, hist_lp_full, model_lp_full = train_linear_probe(trainval_features_tta, trainval_labels, test_features_tta, test_labels)

lp_params = 2048 * num_classes + num_classes
benchmark_resnet_results["2. ResNet-50 (Linear Probe) | Raw Image (Full Trainval)"] = {
    "Method": "ResNet-50 (Linear Probe)", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_lp_raw, "Trainable_Params": lp_params
}
benchmark_resnet_results["2. ResNet-50 (Linear Probe) | TTA 5-view (9:1 Split)"] = {
    "Method": "ResNet-50 (Linear Probe)", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_lp_90, "Test_Acc": acc_lp_tta_90, "Trainable_Params": lp_params
}
benchmark_resnet_results["2. ResNet-50 (Linear Probe) | TTA 5-view (Full Trainval)"] = {
    "Method": "ResNet-50 (Linear Probe)", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_lp_tta_full, "Trainable_Params": lp_params
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ PHƯƠNG PHÁP 2: RESNET-50 LINEAR PROBE")
print("="*75)
print(f"  2.1. Raw Image (Full Trainval):      {acc_lp_raw:.2f}% (Train time: {t_lp_raw:.2f}s)")
print(f"  2.2. TTA 5-view (9:1 Split):         {acc_lp_tta_90:.2f}% (Val: {val_acc_lp_90:.2f}%, Train time: {t_lp_90:.2f}s)")
print(f"  2.3. TTA 5-view (Full Trainval):     {acc_lp_tta_full:.2f}% (Train time: {t_lp_full:.2f}s)")
print("="*75)
save_checkpoint('model_lp_raw', model_lp_raw)
save_checkpoint('model_lp_90', model_lp_90)
save_checkpoint('model_lp_full', model_lp_full)

## 5. PHƯƠNG PHÁP 3: RESNET-50 RESIDUAL ADAPTER (RESNET-ADAPTER)

---

Tương tự cơ chế **CLIP-Adapter**, thiết kế mạng nơ-ron thích ứng dạng cổ chai hai lớp (Bottleneck MLP) với đường nối tắt thặng dư (Residual Connection):
- Tỷ lệ co hẹp (Reduction ratio $r = 4$): Không gian chiếu giảm từ $2048 \to 512 \to 2048$.
- Phương trình pha trộn biểu diễn thích ứng:
  $$\mathbf{f}_{\text{adapter}} = \alpha \cdot \text{MLP}(\mathbf{f}) + (1 - \alpha) \cdot \mathbf{f}$$
  $$\text{Logits} = \mathbf{W}_{\text{fc}} \mathbf{f}_{\text{adapter}} + \mathbf{b}$$
- Residual ratio $\alpha = 0.2$ giúp duy trì đặc trưng giàu ngữ nghĩa ban đầu của ImageNet đồng thời điều chỉnh thích nghi với các đặc điểm động vật học.

In [ ]:
class ResNetAdapter(nn.Module):
    def __init__(self, in_features=2048, hidden_dim=512, num_classes=num_classes, alpha=0.2):
        super().__init__()
        self.alpha = alpha
        self.adapter = nn.Sequential(
            nn.Linear(in_features, hidden_dim, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, in_features, bias=False),
            nn.ReLU(inplace=True)
        )
        self.fc = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        f_res = self.adapter(x)
        f_mixed = self.alpha * f_res + (1.0 - self.alpha) * x
        return self.fc(f_mixed)

def train_resnet_adapter(train_feats, train_lbls, test_feats, test_lbls, val_feats=None, val_lbls=None, epochs=RESNET_EPOCHS, lr=1e-3):
    model = ResNetAdapter(num_classes=num_classes).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    train_loader = DataLoader(TensorDataset(train_feats.to(device), train_lbls.to(device)), batch_size=128, shuffle=True)
    best_val_acc = 0.0
    best_weights = None
    
    history = {"loss": [], "val_acc": []}
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x_b, y_b in train_loader:
            optimizer.zero_grad()
            logits = model(x_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        history["loss"].append(total_loss / len(train_loader))
        
        if val_feats is not None and val_lbls is not None:
            model.eval()
            with torch.no_grad():
                val_preds = model(val_feats.to(device)).argmax(dim=-1)
                val_acc = (val_preds == val_lbls.to(device)).float().mean().item() * 100
                history["val_acc"].append(val_acc)
                if val_acc >= best_val_acc:
                    best_val_acc = val_acc
                    best_weights = {k: v.clone() for k, v in model.state_dict().items()}
                    
    train_time = time.time() - start_time
    if best_weights is not None:
        model.load_state_dict(best_weights)
        
    model.eval()
    with torch.no_grad():
        test_preds = model(test_feats.to(device)).argmax(dim=-1)
        test_acc = (test_preds == test_lbls.to(device)).float().mean().item() * 100
        
    return best_val_acc if val_feats is not None else None, test_acc, train_time, history, model

# 3.1. Raw Image (Full Trainval)
_, acc_adp_raw, t_adp_raw, hist_adp_raw, model_adp_raw = train_resnet_adapter(trainval_features_raw, trainval_labels, test_features_raw, test_labels)

# 3.2. TTA 5-view (9:1 Split)
val_acc_adp_90, acc_adp_tta_90, t_adp_90, hist_adp_90, model_adp_90 = train_resnet_adapter(X_tr_tta_90, y_tr_90, test_features_tta, test_labels, X_val_tta_90, y_val_90)

# 3.3. TTA 5-view (Full Trainval)
_, acc_adp_tta_full, t_adp_full, hist_adp_full, model_adp_full = train_resnet_adapter(trainval_features_tta, trainval_labels, test_features_tta, test_labels)

adp_params = sum(p.numel() for p in ResNetAdapter().parameters())
benchmark_resnet_results["3. ResNet-50 (Adapter) | Raw Image (Full Trainval)"] = {
    "Method": "ResNet-50 (Adapter)", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_adp_raw, "Trainable_Params": adp_params
}
benchmark_resnet_results["3. ResNet-50 (Adapter) | TTA 5-view (9:1 Split)"] = {
    "Method": "ResNet-50 (Adapter)", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_adp_90, "Test_Acc": acc_adp_tta_90, "Trainable_Params": adp_params
}
benchmark_resnet_results["3. ResNet-50 (Adapter) | TTA 5-view (Full Trainval)"] = {
    "Method": "ResNet-50 (Adapter)", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_adp_tta_full, "Trainable_Params": adp_params
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ PHƯƠNG PHÁP 3: RESNET-50 RESIDUAL ADAPTER")
print("="*75)
print(f"  3.1. Raw Image (Full Trainval):      {acc_adp_raw:.2f}% (Train time: {t_adp_raw:.2f}s)")
print(f"  3.2. TTA 5-view (9:1 Split):         {acc_adp_tta_90:.2f}% (Val: {val_acc_adp_90:.2f}%, Train time: {t_adp_90:.2f}s)")
print(f"  3.3. TTA 5-view (Full Trainval):     {acc_adp_tta_full:.2f}% (Train time: {t_adp_full:.2f}s)")
print("="*75)
save_checkpoint('model_adp_raw', model_adp_raw)
save_checkpoint('model_adp_90', model_adp_90)
save_checkpoint('model_adp_full', model_adp_full)

## 6. PHƯƠNG PHÁP 4: RESNET-50 COSINE CLASSIFIER (NORMALIZED PROJECTION HEAD)

---

Thay vì tầng tuyến tính đơn thuần với unconstrained logits, mô hình sử dụng **Cosine Classifier** với tham số nhiệt độ $\tau$ có thể học (Learnable Temperature Scale):
$$\text{Logits}_c = \tau \cdot \frac{\mathbf{f}}{\|\mathbf{f}\|_2} \cdot \frac{\mathbf{w}_c}{\|\mathbf{w}_c\|_2}$$
Cơ chế này tương tự như không gian tích vô hướng của CLIP và Tip-Adapter, giúp triệt tiêu hiện tượng mất cân bằng độ dài vector đặc trưng giữa các giống vật nuôi.

In [ ]:
class CosineClassifier(nn.Module):
    def __init__(self, in_features=2048, num_classes=num_classes, init_tau=10.0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.tau = nn.Parameter(torch.tensor(init_tau))
        
    def forward(self, x):
        x_norm = F.normalize(x, dim=-1)
        w_norm = F.normalize(self.weight, dim=-1)
        return self.tau * (x_norm @ w_norm.t())

def train_cosine_classifier(train_feats, train_lbls, test_feats, test_lbls, val_feats=None, val_lbls=None, epochs=RESNET_EPOCHS, lr=5e-3):
    model = CosineClassifier(num_classes=num_classes).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    train_loader = DataLoader(TensorDataset(train_feats.to(device), train_lbls.to(device)), batch_size=128, shuffle=True)
    best_val_acc = 0.0
    best_weights = None
    
    history = {"loss": [], "val_acc": []}
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x_b, y_b in train_loader:
            optimizer.zero_grad()
            logits = model(x_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        history["loss"].append(total_loss / len(train_loader))
        
        if val_feats is not None and val_lbls is not None:
            model.eval()
            with torch.no_grad():
                val_preds = model(val_feats.to(device)).argmax(dim=-1)
                val_acc = (val_preds == val_lbls.to(device)).float().mean().item() * 100
                history["val_acc"].append(val_acc)
                if val_acc >= best_val_acc:
                    best_val_acc = val_acc
                    best_weights = {k: v.clone() for k, v in model.state_dict().items()}
                    
    train_time = time.time() - start_time
    if best_weights is not None:
        model.load_state_dict(best_weights)
        
    model.eval()
    with torch.no_grad():
        test_preds = model(test_feats.to(device)).argmax(dim=-1)
        test_acc = (test_preds == test_lbls.to(device)).float().mean().item() * 100
        
    return best_val_acc if val_feats is not None else None, test_acc, train_time, history, model

# 4.1. Raw Image (Full Trainval)
_, acc_cos_raw, t_cos_raw, hist_cos_raw, model_cos_raw = train_cosine_classifier(trainval_features_raw, trainval_labels, test_features_raw, test_labels)

# 4.2. TTA 5-view (9:1 Split)
val_acc_cos_90, acc_cos_tta_90, t_cos_90, hist_cos_90, model_cos_90 = train_cosine_classifier(X_tr_tta_90, y_tr_90, test_features_tta, test_labels, X_val_tta_90, y_val_90)

# 4.3. TTA 5-view (Full Trainval)
_, acc_cos_tta_full, t_cos_full, hist_cos_full, model_cos_full = train_cosine_classifier(trainval_features_tta, trainval_labels, test_features_tta, test_labels)

cos_params = sum(p.numel() for p in CosineClassifier().parameters())
benchmark_resnet_results["4. ResNet-50 (Cosine Head) | Raw Image (Full Trainval)"] = {
    "Method": "ResNet-50 (Cosine Head)", "Scenario": "Raw Image (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_cos_raw, "Trainable_Params": cos_params
}
benchmark_resnet_results["4. ResNet-50 (Cosine Head) | TTA 5-view (9:1 Split)"] = {
    "Method": "ResNet-50 (Cosine Head)", "Scenario": "TTA 5-view (9:1 Split)", "Val_Acc": val_acc_cos_90, "Test_Acc": acc_cos_tta_90, "Trainable_Params": cos_params
}
benchmark_resnet_results["4. ResNet-50 (Cosine Head) | TTA 5-view (Full Trainval)"] = {
    "Method": "ResNet-50 (Cosine Head)", "Scenario": "TTA 5-view (Full Trainval)", "Val_Acc": None, "Test_Acc": acc_cos_tta_full, "Trainable_Params": cos_params
}

print("="*75)
print("🏆 KẾT QUẢ ĐÁNH GIÁ PHƯƠNG PHÁP 4: RESNET-50 COSINE HEAD")
print("="*75)
print(f"  4.1. Raw Image (Full Trainval):      {acc_cos_raw:.2f}% (Train time: {t_cos_raw:.2f}s)")
print(f"  4.2. TTA 5-view (9:1 Split):         {acc_cos_tta_90:.2f}% (Val: {val_acc_cos_90:.2f}%, Train time: {t_cos_90:.2f}s)")
print(f"  4.3. TTA 5-view (Full Trainval):     {acc_cos_tta_full:.2f}% (Train time: {t_cos_full:.2f}s)")
print("="*75)
save_checkpoint('model_cos_raw', model_cos_raw)
save_checkpoint('model_cos_90', model_cos_90)
save_checkpoint('model_cos_full', model_cos_full)

## 7. ĐO ĐẠC ĐỘ TRỄ SUY DIỄN

Đo trên thiết bị thực tế, đồng bộ CUDA/MPS. Tách head-only và encoder-only; không tính đọc ảnh/tiền xử lý.


In [ ]:

prototypes = F.normalize(torch.stack([F.normalize(X_tr_tta_90[y_tr_90 == c], dim=-1).mean(0) for c in range(num_classes)]), dim=-1).to(device)
torch.save({'prototypes': prototypes.cpu(), 'classes': class_names, 'fingerprint': manifest['fingerprint']}, output_dir / 'prototypes.pt')
predictors = {'ResNet-50 (Prototype)': lambda x: F.normalize(x, dim=-1) @ prototypes.t(),
    'ResNet-50 (Linear Probe)': model_lp_90.eval(),
    'ResNet-50 (Adapter)': model_adp_90.eval(),
    'ResNet-50 (Cosine Head)': model_cos_90.eval()}
histories = {'Linear Probe': hist_lp_90, 'Residual Adapter': hist_adp_90, 'Cosine Head': hist_cos_90}
results = benchmark_resnet_results
encoder = lambda x: backbone(x).flatten(1)


## 8. TỔNG HỢP KẾT QUẢ & TRỰC QUAN HÓA

Bảy hình: accuracy, loss, validation, latency, tham số, confusion matrix, TTA gain từng lớp. Confusion matrix chọn phương pháp theo validation.


In [ ]:
build_report(results, predictors, histories, encoder)


## 9. BẢNG TỔNG KẾT & XUẤT ARTIFACT

Xem bảng được tạo phía trên và `benchmark_report.md`, CSV/JSON, checkpoints, split manifest, class mapping và PNG trong output. So sánh CLIP/ResNet chỉ khi fingerprint trùng khớp. Không dùng trực tiếp checkpoint mới với backend 37 lớp khi chưa cập nhật ánh xạ lớp và cấu hình inference.
